# Patrón Creacional: Singleton

## Introducción
El patrón Singleton garantiza que una clase tenga una única instancia y proporciona un punto de acceso global a ella. Es útil cuando se necesita un único objeto para coordinar acciones en todo el sistema (por ejemplo, una conexión a base de datos o un registro de logs).

## Objetivos
- Comprender el propósito y la implementación del patrón Singleton.
- Identificar cuándo es útil y cuándo evitarlo.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: App de Estación Meteorológica**
Supón que una app meteorológica necesita un único objeto para acceder a la configuración global de la estación (ubicación, unidades, etc.). El Singleton asegura que solo exista una instancia de la configuración en toda la aplicación.

**¿Dónde se usa en proyectos reales?**
En sistemas de configuración global, gestión de conexiones a base de datos, registro de logs, controladores de hardware, etc.

### Sin patrón Singleton (forma errónea)
Cada vez que se crea un Logger, se obtiene una nueva instancia, lo que puede causar inconsistencias y problemas de sincronización.

In [4]:
import uuid
class Logger:

    def __init__(self):
        self.id = uuid.uuid4()

    def log(self, msg):
        print(f'LOG: {msg} - ID: {self.id}')

logger1 = Logger()
logger1.log('Primer log')


logger2 = Logger()
logger2.log('Segundo log')

print(logger1 is logger2)

LOG: Primer log - ID: 140066ec-bb7e-42ff-9092-717560155658
LOG: Segundo log - ID: 19e5d8a8-a0c3-438d-b446-d584ba199f33
False


### Con patrón Singleton (forma correcta)
El cliente siempre obtiene la misma instancia, asegurando consistencia y acceso global.

In [20]:
import uuid

class SingletonLogger:
    _instance = None
    id = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.id = uuid.uuid4()
        return cls._instance

    def log(self, msg):
        print(f'SINGLETON LOG: {msg} - ID: {self.id}')

logger1 = SingletonLogger()
logger1.log('Primer log')


logger2 = SingletonLogger()
logger2.log('Segundo log')

print(logger1 is logger2)

SINGLETON LOG: Primer log - ID: 5b14e8cb-40c4-4cec-b705-41209c87caf1
SINGLETON LOG: Segundo log - ID: 5b14e8cb-40c4-4cec-b705-41209c87caf1
True


## UML del patrón Singleton
```plantuml
@startuml
class SingletonLogger {
    - _instance: SingletonLogger
    + __new__()
    + log(msg)
}
@enduml
```

## Otro ejemplo de la vida real: Pool de conexiones a base de datos
**Contexto:** una API con varios módulos (usuarios, pedidos, pagos) necesita hablar con la base de datos. Cada conexión real es costosa de abrir y la base de datos tiene un límite de conexiones concurrentes (`max_connections`). Si cada módulo crea su propio pool de conexiones "por si acaso", la aplicación termina abriendo muchas más conexiones de las necesarias y puede agotar el límite de la base de datos.

### Sin patrón (forma errónea)
Cada módulo instancia su propio pool, sin saber que otros módulos ya crearon el suyo.

In [ ]:
import uuid

class PoolConexiones:
    def __init__(self, tamano=5):
        self.id = uuid.uuid4()
        self.tamano = tamano
        self.conexiones_activas = 0
    def obtener_conexion(self):
        self.conexiones_activas += 1
        print(f'Pool {self.id} -> conexión activa #{self.conexiones_activas} (capacidad {self.tamano})')

# El módulo de usuarios y el de pedidos crean pools distintos sin coordinarse
pool_usuarios = PoolConexiones()
pool_pedidos = PoolConexiones()

pool_usuarios.obtener_conexion()
pool_pedidos.obtener_conexion()

print(pool_usuarios is pool_pedidos)

### Con patrón (forma correcta)
Sin importar cuántos módulos pidan el pool, `__new__` siempre devuelve la misma instancia: un único conjunto de conexiones compartido por toda la aplicación.

In [ ]:
import uuid

class PoolConexionesSingleton:
    _instance = None

    def __new__(cls, tamano=5):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.id = uuid.uuid4()
            cls._instance.tamano = tamano
            cls._instance.conexiones_activas = 0
        return cls._instance

    def obtener_conexion(self):
        self.conexiones_activas += 1
        print(f'Pool {self.id} -> conexión activa #{self.conexiones_activas} (capacidad {self.tamano})')


pool_usuarios = PoolConexionesSingleton()
pool_pedidos = PoolConexionesSingleton()

pool_usuarios.obtener_conexion()
pool_pedidos.obtener_conexion()

print(pool_usuarios is pool_pedidos)

### UML del ejemplo de pool de conexiones
```plantuml
@startuml
class PoolConexionesSingleton {
    - _instance: PoolConexionesSingleton
    - id
    - tamano
    - conexiones_activas
    + __new__(tamano)
    + obtener_conexion()
}
@enduml
```

### ¿Dónde más se usa Singleton?
- **Pools de conexiones (DB, Redis, HTTP):** exactamente este ejemplo — frameworks como SQLAlchemy o los clientes de Redis mantienen un único pool compartido por proceso.
- **Registro central de logging:** el `logging.getLogger()` de Python devuelve la misma instancia de logger para un nombre dado en toda la aplicación.
- **Gestor de feature flags / configuración remota:** una única instancia sincronizada con el servicio de configuración, consultada desde cualquier parte del código.
- **Caché en memoria de una aplicación:** un solo objeto de caché compartido, para que dos módulos no mantengan copias desincronizadas de los mismos datos.
- **Colas de trabajos en segundo plano:** un único "worker manager" que coordina todos los jobs, evitando que dos instancias procesen la misma cola en paralelo de forma descoordinada.

**Ejercicio de reflexión:** en aplicaciones con múltiples procesos (ej. varios workers de Gunicorn), el Singleton solo garantiza una única instancia *por proceso*, no por servidor. ¿Qué mecanismo usarías para compartir el estado entre procesos (pista: piensa en Redis o una base de datos)?

## Actividad
Implementa tu propio Singleton para una clase de configuración global.

---

## Explicación de conceptos clave
- **Instancia única:** Singleton asegura que solo exista una instancia de la clase en todo el sistema.
- **Acceso global:** Permite acceder a la instancia desde cualquier parte de la aplicación.
- **Aplicación en la vida real:** Útil en sistemas de configuración, registro de logs, controladores de hardware, etc.

## Conclusión
El patrón Singleton es fundamental cuando se requiere una única instancia de una clase para coordinar acciones globales. Es común en aplicaciones meteorológicas, sistemas de configuración y gestión de recursos compartidos. Debe usarse con precaución para evitar problemas de pruebas y acoplamiento excesivo.